# A2 — InstructPix2Pix trên Google Colab A100
So sánh pretrained và fine-tune U-Net cho chỉnh sửa ảnh bằng câu lệnh. A2.1 đã nộp DeepLabv3; việc đổi đề tài chưa có xác nhận giảng viên đồng ý.

Notebook dùng tập 1.000 mẫu đa dạng để bắt đầu, **chưa phải tập riêng về thời tiết**. Chỉ train U-Net; VAE/text encoder đóng băng. Code Diffusers v0.35.1 được chỉnh để giữ U-Net 8 kênh và đọc split cục bộ. AI hỗ trợ triển khai; chưa chạy GPU thật.


## 1. Mount Drive và cài phụ thuộc
Chọn Runtime → Change runtime type → A100 GPU. Sửa ROOT nếu thư mục của bạn khác. Không cần tài khoản W&B, không upload model lên Hub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, subprocess, json, time, shutil
ROOT = Path('/content/drive/MyDrive/Thac si/Hoc ki 3/Hoc sau va ung dung/Exercises/SDH_HK261_CO5085/A2')
SOURCE = ROOT/'instructpix2pix'
assert (SOURCE/'train_instruct_pix2pix.py').is_file(), SOURCE
assert (SOURCE/'requirements.txt').is_file(), SOURCE/'requirements.txt'
print('Python:', sys.version)
install = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(SOURCE/'requirements.txt')],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(install.stdout)
if install.returncode:
    raise RuntimeError('Cài thư viện thất bại. Gửi các dòng ERROR trong log pip phía trên.')
# Copy code locally; data/cache also stay on local VM storage.
LOCAL = Path('/content/a2_instructpix2pix')
shutil.copytree(SOURCE, LOCAL, dirs_exist_ok=True)
os.chdir(LOCAL)
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'


## 2. GPU và cấu hình
Khởi đầu 256×256, batch 4, accumulation 4: effective batch 16 trên một GPU. Nếu OOM, giảm batch còn 2 và tăng accumulation lên 8. Checkpoint full U-Net/optimizer khá lớn; chuẩn bị hàng chục GB trống trên Drive. Giữ nguyên cấu hình khi resume.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Cần GPU CUDA'
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory/2**30, 'GiB')
MODEL = 'timbrooks/instruct-pix2pix'
RESOLUTION = 256
BATCH = 4
ACCUMULATION = 4
PRECISION = 'bf16' if torch.cuda.is_bf16_supported() else 'fp16'
RUN_ID = 'ip2p_v1'  # Đổi tên để bắt đầu thí nghiệm độc lập.
RUN = ROOT/'runs'/RUN_ID
RUN.mkdir(parents=True, exist_ok=True)
DATA = Path('/content/ip2p_data')
print('Effective batch:', BATCH*ACCUMULATION)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


## 3. Tải dữ liệu và split
Nhóm các bản chỉnh sửa có cùng ảnh gốc bằng SHA-256 ảnh RGB, chia khoảng 80/10/10 theo nhóm. Manifest lưu trên Drive; khi Colab ngắt, tải lại và kiểm tra manifest. Split này chỉ ngăn rò rỉ của lần fine-tune: checkpoint pretrained có thể đã học các mẫu này. Không coi đây là đánh giá độc lập tuyệt đối.

In [ ]:
if not DATA.exists():
    subprocess.run([sys.executable, 'prepare_data.py', '--output', str(DATA)], check=True)
manifest = json.loads((DATA/'split_manifest.json').read_text())
manifest_path = RUN/'split_manifest.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, 'Dữ liệu/split đã đổi; tạo run khác'
else:
    manifest_path.write_text(json.dumps(manifest, indent=2))
print(manifest['counts'])
from datasets import load_from_disk
from IPython.display import display
preview = load_from_disk(str(DATA))['validation'][0]
print(preview['edit_prompt'])
display(preview['input_image'], preview['edited_image'])


## 4. Fine-tune thử 100 optimizer steps
Dùng learning rate 1e-5 để bắt đầu điều chỉnh checkpoint đã học chỉnh ảnh. Không flip ngẫu nhiên vì lệnh có thể chứa trái/phải. Checkpoint mỗi 100 steps, giữ hai checkpoint gần nhất. Log loss dùng TensorBoard. Thời gian pilot gồm khởi tạo/lưu model nên chỉ là ước tính thô; xem thêm tốc độ it/s trong progress bar.

In [ ]:
def train_to(max_steps):
    config = dict(model=MODEL, resolution=RESOLUTION, batch=BATCH, accumulation=ACCUMULATION,
                  precision=PRECISION, learning_rate=1e-5, seed=42)
    config_path = RUN/'experiment.json'
    if config_path.exists():
        assert json.loads(config_path.read_text()) == config, 'Không đổi config khi resume; tạo RUN_ID khác'
    else:
        config_path.write_text(json.dumps(config, indent=2))
    checkpoints = sorted(RUN.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]))
    if checkpoints and int(checkpoints[-1].name.split('-')[-1]) >= max_steps:
        print('Đã đạt', max_steps, 'steps. Không train lại.'); return
    command = [sys.executable, 'train_instruct_pix2pix.py',
               '--pretrained_model_name_or_path', MODEL, '--dataset_name', str(DATA),
               '--output_dir', str(RUN), '--resolution', str(RESOLUTION), '--center_crop',
               '--train_batch_size', str(BATCH), '--gradient_accumulation_steps', str(ACCUMULATION),
               '--gradient_checkpointing', '--mixed_precision', PRECISION,
               '--max_train_steps', str(max_steps), '--learning_rate', '1e-5',
               '--lr_scheduler', 'constant', '--lr_warmup_steps', '0',
               '--conditioning_dropout_prob', '0.05', '--checkpointing_steps', '100',
               '--checkpoints_total_limit', '2', '--dataloader_num_workers', '2',
               '--report_to', 'tensorboard', '--seed', '42']
    if checkpoints:
        command += ['--resume_from_checkpoint', 'latest']
    (RUN/f'command_{max_steps}.json').write_text(json.dumps(command, indent=2))
    with (RUN/f'train_{max_steps}.log').open('a') as log:
        start = time.perf_counter()
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        if process.wait():
            raise RuntimeError('Training failed; xem log trên Drive. Nếu OOM, tạo run mới với batch nhỏ hơn.')
    seconds = time.perf_counter()-start
    (RUN/f'timing_{max_steps}.json').write_text(json.dumps(dict(seconds=seconds, target_steps=max_steps)))
    print('Tổng thời gian lần gọi:', seconds/60, 'phút')
train_to(100)


## 5. So sánh validation sau pilot
Mỗi hàng ảnh: input | target tham chiếu | output mô hình. Cùng seed và guidance cho hai mô hình. Hai metric là CLIP image-target cosine (cao hơn tốt hơn) và paired pixel MAE (thấp hơn tốt hơn), chỉ là proxy so với một đáp án; không phải điểm làm đúng lệnh. Ảnh chỉnh hợp lệ có thể khác target. Giữ safety checker và ghi số ảnh bị đánh dấu.

In [ ]:
def compare(label, unet=None, split='validation', limit=20):
    output = RUN/'evaluation'/f'{split}_{label}'
    if (output/'summary.json').exists():
        print((output/'summary.json').read_text()); return
    command = [sys.executable, 'compare.py', '--data', str(DATA), '--model', MODEL,
               '--output', str(output), '--split', split, '--limit', str(limit),
               '--resolution', str(RESOLUTION), '--steps', '20', '--seed', '123']
    if unet is not None: command += ['--unet', str(unet)]
    subprocess.run(command, check=True)
compare('pretrained')
compare('step100', RUN/'checkpoint-100')
from IPython.display import Image
for label in ['pretrained', 'step100']:
    print(label)
    display(Image(filename=str(RUN/'evaluation'/f'validation_{label}'/'0000_input_target_prediction.png')))


## 6. Tiếp tục tới 2.000 steps
Đánh giá pilot trước khi chạy tiếp. Chạy ô này khi muốn tiếp tục. Resume từ checkpoint gần nhất trên Drive; nếu ngắt giữa chừng, chạy lại các ô setup/data rồi ô này. Không sửa cấu hình hay split của run. Checkpoint trước 2.000 có thể bị dọn theo giới hạn hai bản.
Resume khôi phục model/optimizer; không cam kết bit-for-bit. Kiểm tra log RNG khi resume trên phiên Colab.


In [ ]:
MAX_STEPS = 2000
train_to(MAX_STEPS)
compare(f'step{MAX_STEPS}', RUN/f'checkpoint-{MAX_STEPS}')


## 7. Loss và chọn checkpoint bằng validation
Chọn pretrained hoặc checkpoint dựa trên validation và chất lượng ảnh. Không dùng test để chọn hyperparameter. Kiểm tra đúng lệnh, bảo toàn nội dung, artefact theo rubric 1–5; log MAE/CLIP chưa đủ để kết luận.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUN}/logs


## 8. Đánh giá test cuối
Chỉ chạy sau khi chốt lựa chọn bằng validation. Test tối đa 100 mẫu; script tự giới hạn theo số mẫu có sẵn. Báo số mẫu thực tế, seed, độ phân giải và độ trùng có thể có với pretrained training data.

In [ ]:
SELECTED_UNET = RUN/f'checkpoint-{MAX_STEPS}'  # Chọn bằng validation; có thể dùng RUN để lấy model cuối.
assert (SELECTED_UNET/'unet').is_dir()
compare('pretrained', split='test', limit=100)
compare('selected', SELECTED_UNET, split='test', limit=100)


## 9. Demo trên ảnh của bạn
Chạy sau khi training/evaluation xong để GPU không giữ hai bản model cùng lúc. Kết quả lưu trên Drive. Dùng lệnh tiếng Anh để phù hợp dữ liệu gốc.

In [ ]:
from diffusers import StableDiffusionInstructPix2PixPipeline, UNet2DConditionModel
from PIL import Image as PILImage, ImageOps
from google.colab import files
uploaded = files.upload()
filename = next(iter(uploaded))
source = ImageOps.fit(PILImage.open(filename).convert('RGB'), (RESOLUTION, RESOLUTION))
PROMPT = 'make it a day light photograph'
dtype = torch.bfloat16 if PRECISION == 'bf16' else torch.float16
pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(MODEL, torch_dtype=dtype)
pipe.unet = UNet2DConditionModel.from_pretrained(SELECTED_UNET, subfolder='unet', torch_dtype=dtype)
pipe.to('cuda')
result = pipe(PROMPT, image=source, num_inference_steps=30, guidance_scale=7.5,
              image_guidance_scale=1.5, generator=torch.Generator('cuda').manual_seed(123))
display(source, result.images[0])
result.images[0].save(RUN/'demo.png')
del pipe
torch.cuda.empty_cache()


## Báo cáo và nguồn
- Paper: Brooks, Holynski, Efros. *InstructPix2Pix: Learning to Follow Image Editing Instructions*, CVPR 2023. https://arxiv.org/abs/2211.09800
- Code tác giả: https://github.com/timothybrooks/instruct-pix2pix
- Script train được chỉnh từ Diffusers v0.35.1 (Apache-2.0): https://github.com/huggingface/diffusers/blob/v0.35.1/examples/instruct_pix2pix/train_instruct_pix2pix.py
- Dataset nhỏ: https://huggingface.co/datasets/fusing/instructpix2pix-1000-samples

Báo rõ dữ liệu tổng hợp, giới hạn 1k mẫu, nguy cơ pretrained overlap, cấu hình, số steps, train loss, proxy metrics và rubric con người. Không gọi thí nghiệm này là tái hiện full paper hoặc khẳng định fine-tune sẽ cải thiện. Checkpoint/model/dataset không commit Git. Nội dung và code có hỗ trợ OpenAI Codex; nhóm chịu trách nhiệm kiểm chứng.
